In [1]:
import numpy as np
import pandas as pd 
import os
import re
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
from sklearn.preprocessing import minmax_scale
import IPython.display as ipd
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, recall_score

In [2]:
import pandas as pd
dataf = pd.read_csv('MFCCoutput.csv')
dataf = dataf.drop(columns=dataf.columns[0], axis = 1)
dataf.head()

,0,1,2,3,4,5,6,7,8,9,...,119,120,121,122,123,124,125,126,127,class
0,-398.97055,92.846680,-6.212745,19.836878,-3.015201,9.478265,5.134083,6.716100,1.437088,0.016323,...,0.018726,-0.244671,-0.650705,-0.262222,0.137795,-0.539087,-0.257223,-0.230522,-0.188446,1.0
1,-232.39255,115.044525,-21.028315,39.190132,-17.016842,7.619745,-2.724971,4.140653,-1.230497,-0.977595,...,0.197640,0.564010,0.166091,-0.022294,0.257573,0.168772,0.264451,-0.257519,-0.390595,1.0
2,-466.48450,89.272385,-8.458461,30.776363,-11.168960,18.305796,0.989266,10.417193,0.574027,1.563966,...,0.232031,0.074036,0.237405,0.154122,0.281900,0.768758,-0.073312,0.096406,0.285212,1.0
3,-466.73505,62.805060,12.439709,29.304922,12.614448,9.676723,1.418015,13.074185,0.037665,3.573357,...,0.408432,-0.018611,-0.274066,0.071348,0.231881,0.456909,0.000649,-0.379244,-0.105873,1.0
4,-426.44970,80.985410,-4.792509,36.350883,-0.092283,19.156744,-5.060070,10.123955,2.196594,1.254182,...,-0.064134,0.120973,0.070115,0.138273,0.084424,0.080933,-0.295089,-0.199056,0.120445,1.0


In [3]:
dataf.loc[dataf['class']=='non_dysarthria','class'] = 0.0
dataf.loc[dataf['class']=='dysarthria','class'] = 1.0
dataf['class'] = dataf['class'].astype(float)

X = dataf.iloc[:,:-1].values
y = dataf.iloc[:,-1]

In [4]:
X.shape, y.shape

((4314, 128), (4314,))

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)


In [6]:
print(X_train.shape)
print(X_test.shape)

(3451, 128)
(863, 128)


In [7]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from collections import defaultdict
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier, VotingClassifier

In [8]:
hyperparameters_RFC = {'n_estimators': 600, 'max_depth': 90, 'min_samples_split': 6, 'min_samples_leaf': 3,
                       'max_features': 'sqrt', 'bootstrap': False, 'criterion': 'entropy'}

hyperparameters_XGB = {'max_depth': 9,
                       'min_child_weight': 1,
                       'learning_rate': 0.2,
                       'subsample': 0.8,
                       'colsample_bytree': 1.0,
                       'gamma': 0,
                       'n_estimators': 600,
                       'use_label_encoder': False,
                       'eval_metric': 'rmse',
                       'objective': 'binary:logistic'}

hyperparameters_CB = {'bagging_temperature': 0.8607305832563434, 'bootstrap_type': 'MVS',
                      'colsample_bylevel': 0.917411003148779,
                      'depth': 8, 'grow_policy': 'SymmetricTree', 'iterations': 918, 'l2_leaf_reg': 8,
                      'learning_rate': 0.29287291117375575, 'max_bin': 231, 'min_data_in_leaf': 9, 'od_type': 'Iter',
                      'od_wait': 21, 'one_hot_max_size': 7, 'random_strength': 0.6963042728397884,
                      'scale_pos_weight': 1.924541179848884, 'subsample': 0.6480869299533999}

rf = RandomForestClassifier(**hyperparameters_RFC, random_state=150)
cb = CatBoostClassifier(**hyperparameters_CB)
xgb = XGBClassifier(**hyperparameters_XGB)

In [9]:
print(dataf.head())

           0           1          2          3          4          5  \
0 -398.97055   92.846680  -6.212745  19.836878  -3.015201   9.478265   
1 -232.39255  115.044525 -21.028315  39.190132 -17.016842   7.619745   
2 -466.48450   89.272385  -8.458461  30.776363 -11.168960  18.305796   
3 -466.73505   62.805060  12.439709  29.304922  12.614448   9.676723   
4 -426.44970   80.985410  -4.792509  36.350883  -0.092283  19.156744   

          6          7         8         9  ...       119       120       121  \
0  5.134083   6.716100  1.437088  0.016323  ...  0.018726 -0.244671 -0.650705   
1 -2.724971   4.140653 -1.230497 -0.977595  ...  0.197640  0.564010  0.166091   
2  0.989266  10.417193  0.574027  1.563966  ...  0.232031  0.074036  0.237405   
3  1.418015  13.074185  0.037665  3.573357  ...  0.408432 -0.018611 -0.274066   
4 -5.060070  10.123955  2.196594  1.254182  ... -0.064134  0.120973  0.070115   

        122       123       124       125       126       127  class  
0 -0.2622

In [10]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import ExtraTreesClassifier

# Assuming you have a feature matrix 'X' and target labels 'y'
X = dataf.iloc[:, :-1]  # All columns except the last one
y = dataf.iloc[:, -1]   # Last column as target

# Define classifiers with hyperparameters
cb = CatBoostClassifier()  # Suppress CatBoost logs
lgbm = LGBMClassifier()
et = ExtraTreesClassifier()

# Step 1: Train the classifiers
cb.fit(X, y)
lgbm.fit(X, y)
et.fit(X, y)

# Step 2: Get feature importance scores
cb_importance = cb.get_feature_importance()
lgbm_importance = lgbm.feature_importances_
et_importance = et.feature_importances_

# Step 3: Convert to DataFrame
feature_names = dataf.columns[:-1]  # Assuming last column is 'class'
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'CB_Importance': cb_importance,
    'LGBM_Importance': lgbm_importance,
    'ET_Importance': et_importance
})

# Normalize importance scores 
importance_df[['CB_Importance', 'LGBM_Importance', 'ET_Importance']] = \
    importance_df[['CB_Importance', 'LGBM_Importance', 'ET_Importance']].apply(lambda x: x / np.max(x))

# Compute average importance
importance_df['Avg_Importance'] = importance_df[['CB_Importance', 'LGBM_Importance', 'ET_Importance']].mean(axis=1)

# Sort by Avg_Importance
importance_df = importance_df.sort_values(by='Avg_Importance', ascending=False).reset_index(drop=True)

importance_df.to_csv("feature_importance_basic.csv", index=False)

# # Step 7: Assign Fibonacci weights
# def fibonacci(n):
#     fib_series = [1, 1]
#     for _ in range(n-2):
#         fib_series.append(fib_series[-1] + fib_series[-2])
#     return fib_series

# fib_weights = fibonacci(len(importance_df))

# # Compute final ranking score
# importance_df['Fibonacci_Weight'] = fib_weights
# importance_df['Final_Rank_Score'] = importance_df['Avg_Importance'] * importance_df['Fibonacci_Weight']

# # Sort based on Final Rank Score
# importance_df = importance_df.sort_values(by='Final_Rank_Score', ascending=False).reset_index(drop=True)

# # Top ranked features
# print(importance_df[['Feature', 'Final_Rank_Score']].head(20))


Learning rate set to 0.019232
0:	learn: 0.6571122	total: 167ms	remaining: 2m 46s
1:	learn: 0.6288094	total: 214ms	remaining: 1m 46s
2:	learn: 0.6072127	total: 245ms	remaining: 1m 21s
3:	learn: 0.5862682	total: 313ms	remaining: 1m 18s
4:	learn: 0.5671602	total: 534ms	remaining: 1m 46s
5:	learn: 0.5487795	total: 565ms	remaining: 1m 33s
6:	learn: 0.5314452	total: 594ms	remaining: 1m 24s
7:	learn: 0.5141469	total: 621ms	remaining: 1m 16s
8:	learn: 0.4976034	total: 647ms	remaining: 1m 11s
9:	learn: 0.4763892	total: 675ms	remaining: 1m 6s
10:	learn: 0.4610827	total: 704ms	remaining: 1m 3s
11:	learn: 0.4443572	total: 734ms	remaining: 1m
12:	learn: 0.4310225	total: 770ms	remaining: 58.5s
13:	learn: 0.4156707	total: 802ms	remaining: 56.5s
14:	learn: 0.4034640	total: 840ms	remaining: 55.1s
15:	learn: 0.3913176	total: 875ms	remaining: 53.8s
16:	learn: 0.3811392	total: 906ms	remaining: 52.4s
17:	learn: 0.3717761	total: 934ms	remaining: 50.9s
18:	learn: 0.3628075	total: 964ms	remaining: 49.8s
19:	l

In [29]:
ranked_features=importance_df

In [11]:
importance_df = pd.read_csv('feature_importance_basic.csv')
print(importance_df.head())

   Feature  CB_Importance  LGBM_Importance  ET_Importance  Avg_Importance
0        0       1.000000         1.000000       1.000000        1.000000
1       36       0.642909         0.740157       0.766617        0.716561
2       82       0.484306         0.732283       0.782581        0.666390
3       49       0.290911         0.787402       0.672607        0.583640
4       12       0.293238         0.464567       0.949464        0.569090


In [12]:
print(importance_df.isna().sum())


Feature            0
CB_Importance      0
LGBM_Importance    0
ET_Importance      0
Avg_Importance     0
dtype: int64
